In [1]:
import cv2
import mediapipe as mp
import numpy as np
import os
from pathlib import Path

from utils.camera import load_cam_infos

In [2]:
def enhance_image_with_blending(image, roi_points, brightness_factor=1.3):
    """
    Enhance image brightness outside ROI and blend with original ROI using Laplacian pyramid.
    """
    # Create masks
    mask = np.zeros(image.shape[:2], dtype=np.uint8)
    roi_points = np.array(roi_points, dtype=np.int32)
    cv2.fillPoly(mask, [roi_points], 255)
    
    # Create enhanced version of the image
    enhanced = cv2.convertScaleAbs(image, alpha=brightness_factor, beta=0)
    
    # Initialize output image
    result = np.zeros_like(image)
    
    # Number of pyramid levels
    levels = 4
    
    # Generate Gaussian pyramid for mask
    mask_pyramid = [mask.astype(float) / 255]
    for i in range(levels-1):
        mask_pyramid.append(cv2.pyrDown(mask_pyramid[-1]))
    
    # Generate Laplacian pyramids for images
    orig_pyramid = [image.astype(float)]
    enhanced_pyramid = [enhanced.astype(float)]
    
    for i in range(levels-1):
        orig_pyramid.append(cv2.pyrDown(orig_pyramid[-1]))
        enhanced_pyramid.append(cv2.pyrDown(enhanced_pyramid[-1]))
    
    # Create Laplacian pyramids
    orig_laplacian = []
    enhanced_laplacian = []
    
    for i in range(levels-1):
        orig_size = (orig_pyramid[i].shape[1], orig_pyramid[i].shape[0])
        enhanced_size = (enhanced_pyramid[i].shape[1], enhanced_pyramid[i].shape[0])
        
        orig_up = cv2.pyrUp(orig_pyramid[i+1], dstsize=orig_size)
        enhanced_up = cv2.pyrUp(enhanced_pyramid[i+1], dstsize=enhanced_size)
        
        orig_laplacian.append(orig_pyramid[i] - orig_up)
        enhanced_laplacian.append(enhanced_pyramid[i] - enhanced_up)
    
    orig_laplacian.append(orig_pyramid[-1])
    enhanced_laplacian.append(enhanced_pyramid[-1])
    
    # Blend pyramids using mask
    blended_pyramid = []
    for orig_lap, enhanced_lap, mask_g in zip(orig_laplacian, enhanced_laplacian, mask_pyramid):
        blended = orig_lap * mask_g[..., np.newaxis] + enhanced_lap * (1 - mask_g[..., np.newaxis])
        blended_pyramid.append(blended)
    
    # Reconstruct image
    result = blended_pyramid[-1]
    for i in range(levels-2, -1, -1):
        size = (blended_pyramid[i].shape[1], blended_pyramid[i].shape[0])
        result = cv2.pyrUp(result, dstsize=size)
        result += blended_pyramid[i]
    
    return np.clip(result, 0, 255).astype(np.uint8)

def enhance_image(image, alpha=2.3, beta=15):
    """Enhanced image preprocessing with multiple techniques"""
    # Convert to LAB color space
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    
    # Apply CLAHE to L channel
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    cl = clahe.apply(l)
    
    # Merge channels
    enhanced_lab = cv2.merge((cl,a,b))
    
    # Convert back to BGR
    enhanced_bgr = cv2.cvtColor(enhanced_lab, cv2.COLOR_LAB2BGR)
    
    # Increase contrast
    enhanced_contrast = cv2.convertScaleAbs(enhanced_bgr, alpha=alpha, beta=beta)
    
    return enhanced_contrast

def process_hand_landmarks(hand_landmarks, handedness, image_shape):
    """Convert MediaPipe hand landmarks to a list of [x, y, confidence] format"""
    h, w = image_shape[:2]
    keypoints = []
    
    for landmark in hand_landmarks.landmark:
        # Convert normalized coordinates to pixel coordinates
        x = landmark.x * w
        y = landmark.y * h
        # MediaPipe provides confidence per hand, not per keypoint
        confidence = 1.0
        keypoints.extend([float(x), float(y), confidence])
    
    return keypoints

def detect_hands(hands, image, save_enhanced=False, folder_name="enhanced", index=0):
    """Attempt hand detection with various image enhancements"""
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = hands.process(image_rgb)
    
    if results.multi_hand_landmarks and len(results.multi_hand_landmarks) == 2:
        return results, image, "original"
    
    # First enhancement: Basic image enhancement
    enhanced_img = enhance_image(image)
    enhanced_rgb = cv2.cvtColor(enhanced_img, cv2.COLOR_BGR2RGB)
    results_enhanced = hands.process(enhanced_rgb)
    
    if results_enhanced.multi_hand_landmarks and len(results_enhanced.multi_hand_landmarks) == 2:
        if save_enhanced:
            cv2.imwrite(f'{folder_name}/enhanced_success_{index}.jpg', enhanced_img)
        return results_enhanced, enhanced_img, "enhanced"
    
    # Second enhancement: Try with ROI blending if we detected at least one hand
    if results.multi_hand_landmarks and len(results.multi_hand_landmarks) == 1:
        # Get ROI from the detected hand
        hand_landmarks = results.multi_hand_landmarks[0]
        h, w = image.shape[:2]
        points = [(int(landmark.x * w), int(landmark.y * h)) for landmark in hand_landmarks.landmark]
        x_coords, y_coords = zip(*points)
        roi_points = np.array([
            [min(x_coords) - 20, min(y_coords) - 20],
            [min(x_coords) - 20, max(y_coords) + 20],
            [max(x_coords) + 20, max(y_coords) + 20],
            [max(x_coords) + 20, min(y_coords) - 20]
        ], dtype=np.int32)
        
        blended_img = enhance_image_with_blending(image, roi_points)
        blended_rgb = cv2.cvtColor(blended_img, cv2.COLOR_BGR2RGB)
        results_blended = hands.process(blended_rgb)
        
        if results_blended.multi_hand_landmarks and len(results_blended.multi_hand_landmarks) == 2:
            if save_enhanced:
                cv2.imwrite(f'{folder_name}/blended_success_{index}.jpg', blended_img)
            return results_blended, blended_img, "blended"
    
    # Return best result (prefer more hands detected)
    if results_enhanced.multi_hand_landmarks and len(results_enhanced.multi_hand_landmarks) > len(results.multi_hand_landmarks or []):
        return results_enhanced, enhanced_img, "enhanced"
    return results, image, "original"

In [9]:
conf = 0.5
img_path = "data/input/tony/Marshall/camera05/images/color_002191_camera01.jpg"
img = cv2.imread(img_path)
cam_params = load_cam_infos(Path("./data/input/tony/"), orbbec=False)[f'camera05']

if img is None:
    print(f"Failed to load image: {img_path}")

img = cv2.undistort(
    img, 
    cam_params['intrinsics'], 
    np.array([cam_params['radial_params'][0]] + [cam_params['radial_params'][1]] + list(cam_params['tangential_params'][:2]) + [cam_params['radial_params'][2]] + [0, 0, 0])
)

NameError: name 'Path' is not defined

In [ ]:
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=2,
    min_detection_confidence=conf
)

results, processed_image, method_used = detect_hands(hands, img, save_enhanced=True, folder_name="enhanced", index=0)

I0000 00:00:1746620450.244914  957284 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4 Max
W0000 00:00:1746620450.250427  960557 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1746620450.254161  960557 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [21]:
image_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
results = hands.process(image_rgb)

In [52]:
from openpose_impl import model
from openpose_impl import util
from openpose_impl.body import Body
from openpose_impl.hand import Hand
import copy

BASE_PTH_MODELS = './openpose_impl/model/'
body_estimation = Body(BASE_PTH_MODELS + 'body_pose_model.pth')
hand_estimation = Hand(BASE_PTH_MODELS + 'hand_pose_model.pth')


In [53]:
img_rotated = cv2.rotate(img, cv2.ROTATE_180)
candidate, subset = body_estimation(img_rotated)
hands_list = util.handDetect(candidate, subset, img_rotated)

print(hands_list)
data = {
            "people": [{
                "hand_left_keypoints_2d": [],
                "hand_right_keypoints_2d": []
            }]
}
        
# Process detected hands
if hands_list:
    num_hands = len(hands_list)
    image_for_drawing = processed_image.copy()
    all_hand_peaks = []
    # Process each detected hand
    for hand_idx, (x, y, w, is_left) in enumerate(hands_list):
        # Extract and process hand region
        hand_roi = processed_image[y:y+w, x:x+w, :]
        peaks = hand_estimation(hand_roi)
        
        # Adjust peaks to image space
        peaks[:, 0] = np.where(peaks[:, 0]==0, peaks[:, 0], peaks[:, 0]+x)
        peaks[:, 1] = np.where(peaks[:, 1]==0, peaks[:, 1], peaks[:, 1]+y)
        
        all_hand_peaks.append(peaks)
        # Convert peaks to keypoints
        hand_keypoints = []
        for i, keypoint in enumerate(peaks):
            x_coord, y_coord = keypoint
            hand_keypoints.extend([float(x_coord), float(y_coord), 1.0])
        
        # Store keypoints based on handedness
        if is_left:
            data["people"][0]["hand_left_keypoints_2d"] = hand_keypoints
        else:
            data["people"][0]["hand_right_keypoints_2d"] = hand_keypoints


canvas = util.draw_handpose(copy.deepcopy(img), all_hand_peaks)
cv2.imshow(canvas)
cv2.waitKey(0)
cv2.destroyAllWindows()

[[1022, 762, 272, True], [773, 646, 339, False]]


error: OpenCV(4.11.0) :-1: error: (-5:Bad argument) in function 'imshow'
> Overload resolution failed:
>  - imshow() missing required argument 'mat' (pos 2)
>  - imshow() missing required argument 'mat' (pos 2)
>  - imshow() missing required argument 'mat' (pos 2)


In [54]:
alpha = 0

while alpha < 3.1:
    beta = 0
    while beta < 25:
        enhanced_img = enhance_image(img, alpha=alpha, beta=beta)
        enhanced_rgb = cv2.cvtColor(enhanced_img, cv2.COLOR_BGR2RGB)
        results_enhanced = hands.process(enhanced_rgb)
        if results_enhanced.multi_hand_landmarks:
            print(f"For alpha: {alpha} and beta: {beta} found {len(results_enhanced.multi_hand_landmarks)}...")
        angles = [0, 90, 180, 270]
        for angle in angles:
            if angle == 0:
                img_rotated = enhanced_rgb
            else:
                # For 90 degree rotations, use cv2's built-in functions
                if angle == 90:
                    img_rotated = cv2.rotate(enhanced_rgb, cv2.ROTATE_90_CLOCKWISE)
                elif angle == 180:
                    img_rotated = cv2.rotate(enhanced_rgb, cv2.ROTATE_180)
                elif angle == 270:
                    img_rotated = cv2.rotate(enhanced_rgb, cv2.ROTATE_90_COUNTERCLOCKWISE)
            
            results_enhanced = hands.process(img_rotated)
            if results_enhanced.multi_hand_landmarks:
                print(f"For MediaPipe alpha: {alpha}, beta: {beta} and rotation: {angle} found {len(results_enhanced.multi_hand_landmarks)}...")
            # else:
            #     candidate, subset = body_estimation(img_rotated)
            #     hands_list = util.handDetect(candidate, subset, img_rotated)
            #     if len(hands_list) > 0:
            #         print(f"For OpenPose alpha: {alpha}, beta: {beta} and rotation: {angle} found {len(hands_list)}...")
        beta += 5 
    alpha += 0.3



W0000 00:00:1746627225.085986  960561 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


For MediaPipe alpha: 1.2, beta: 0 and rotation: 90 found 1...
For MediaPipe alpha: 1.2, beta: 5 and rotation: 90 found 1...
For MediaPipe alpha: 1.5, beta: 15 and rotation: 90 found 1...
For MediaPipe alpha: 1.5, beta: 20 and rotation: 90 found 1...
For MediaPipe alpha: 2.1, beta: 10 and rotation: 180 found 1...
For MediaPipe alpha: 2.1, beta: 15 and rotation: 180 found 1...
For alpha: 2.1 and beta: 20 found 1...
For MediaPipe alpha: 2.1, beta: 20 and rotation: 0 found 1...
For MediaPipe alpha: 2.1, beta: 20 and rotation: 180 found 1...
For alpha: 2.1 and beta: 25 found 1...
For MediaPipe alpha: 2.1, beta: 25 and rotation: 0 found 1...
For MediaPipe alpha: 2.1, beta: 25 and rotation: 180 found 1...
For MediaPipe alpha: 2.1, beta: 30 and rotation: 180 found 1...
For MediaPipe alpha: 2.1, beta: 35 and rotation: 180 found 1...
For MediaPipe alpha: 2.4, beta: 0 and rotation: 180 found 1...
For MediaPipe alpha: 2.4, beta: 5 and rotation: 180 found 1...
For MediaPipe alpha: 2.4, beta: 10 and

In [43]:
results_enhanced.multi_hand_landmarks

In [3]:
import math

def rotate_point(x, y, angle_degrees, clockwise=True):
    angle_radians = math.radians(-angle_degrees if clockwise else angle_degrees)
    cos_theta = math.cos(angle_radians)
    sin_theta = math.sin(angle_radians)
    x_new = x * cos_theta - y * sin_theta
    y_new = x * sin_theta + y * cos_theta
    return x_new, y_new

In [8]:
print(rotate_point(2, 3, 270))

(-3.0000000000000004, 1.9999999999999996)


In [ ]:
import cv2


img_path = "data/input/tony/Marshall/camera05/images/color_002191_camera01.jpg"
img = cv2.imread(img_path)
cam_params = load_cam_infos(Path("./data/input/tony/"), orbbec=False)[f'camera05']

if img is None:
    print(f"Failed to load image: {img_path}")

img = cv2.undistort(
    img, 
    cam_params['intrinsics'], 
    np.array([cam_params['radial_params'][0]] + [cam_params['radial_params'][1]] + list(cam_params['tangential_params'][:2]) + [cam_params['radial_params'][2]] + [0, 0, 0])
)

alpha = 15.0
beta = 500

new_img = cv2.convertScaleAbs(img, alpha=1.0, beta=50)

cv2.imwrite(f"test/test_{alpha}_{beta}.jpg", new_img)

True

: 